In [3]:
from pathlib import Path

# Caminho do diretório base
diretorio = Path('/Volumes/ssd_externo/CEF Edital 2011/Laudos de avaliação simplificados/')

# Buscar recursivamente em todos os subdiretórios usando '**/'
arquivos_pdf_7127 = list(diretorio.glob('**/7127*.pdf'))

# Mostrar resultado
print(f'Total de arquivos PDF que começam com "7127": {len(arquivos_pdf_7127)}')
print(f'\nArquivos encontrados:')
for arquivo in arquivos_pdf_7127:
    # Mostrar caminho relativo ao diretório base
    caminho_relativo = arquivo.relative_to(diretorio)
    print(f'  - {caminho_relativo}')

Total de arquivos PDF que começam com "7127": 480

Arquivos encontrados:
  - AGE 0380/7127.0380.481407_2012.01.01.01/7127.0380.481407_2012.01.01.01.pdf
  - AGE 0384/7127.0384.013526_2012.01.01.01/7127.0384.013526_2012.01.01.01.pdf
  - AGE 0384/7127.0384.045150_2012.01.01.01/7127.0384.045150_2012.01.01.01.pdf
  - AGE 0384/7127.0384.083359_2012.01.01.01/7127.0384.083359_2012.01.01.01.pdf
  - AGE 0384/7127.0384.875116_2011.01.01.01/7127.0384.875116_2011.01.01.01.pdf
  - AGE 0394/7127.0394.017404_2012.01.01.01/7127.0394.017404_2012.01.01.01.pdf
  - AGE 0394/7127.0394.049584_2014.01.01.01/7127.0394.049584_2014.01.01.01.pdf
  - AGE 0394/7127.0394.067866_2012.01.01.01/7127.0394.067866_2012.01.01.01.pdf
  - AGE 0394/7127.0394.083364_2012.01.01.01/7127.0394.083364_2012.01.01.01.pdf
  - AGE 0394/7127.0394.091883_2012.01.01.01/7127.0394.091883_2012.01.01.01.pdf
  - AGE 0394/7127.0394.1005821_2013.01.01.01/7127.0394.1005821_2013.01.01.01.pdf
  - AGE 0394/7127.0394.171253_2013.01.01.01/7127.0394.17

## Extrair imagens de arquivos XLSX

Código alternativo para extrair imagens diretamente dos arquivos Excel originais.

In [ ]:
from openpyxl import load_workbook
from pathlib import Path
from PIL import Image
import io

# Configurações
diretorio_base = Path('/Volumes/ssd_externo/CEF Edital 2011/Laudos de avaliação simplificados/')
output_dir = Path('/Users/fjcosta/Downloads/imagens_extraidas_todos_xlsx')

# Criar diretório de saída
output_dir.mkdir(parents=True, exist_ok=True)

# Buscar todos os arquivos XLSX que começam com '7127'
arquivos_xlsx = list(diretorio_base.glob('**/7127*.xlsx'))

print(f'Encontrados {len(arquivos_xlsx)} arquivos XLSX\n')

total_imagens = 0
arquivos_processados = 0
arquivos_com_erro = 0

for xlsx_path in arquivos_xlsx:
    try:
        print(f'📂 Processando: {xlsx_path.name}')
        workbook = load_workbook(xlsx_path)
        
        imagens_arquivo = 0
        
        # Percorrer todas as planilhas
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            
            # Verificar se há imagens na planilha
            if hasattr(sheet, '_images') and sheet._images:
                for img_index, image in enumerate(sheet._images):
                    try:
                        # Extrair dados da imagem
                        image_data = image._data()
                        
                        # Converter para PIL Image
                        pil_image = Image.open(io.BytesIO(image_data))
                        
                        # Detectar formato
                        img_format = pil_image.format.lower() if pil_image.format else 'png'
                        
                        # Nome do arquivo
                        image_name = f"{xlsx_path.stem}_{sheet_name}_img{img_index+1}.{img_format}"
                        # Remover caracteres problemáticos do nome
                        image_name = image_name.replace('/', '_').replace('\\', '_')
                        image_path = output_dir / image_name
                        
                        # Salvar
                        pil_image.save(image_path)
                        
                        imagens_arquivo += 1
                        total_imagens += 1
                    except Exception as e:
                        print(f'  ⚠️ Erro ao extrair imagem: {e}')
        
        workbook.close()
        arquivos_processados += 1
        print(f'  ✓ {imagens_arquivo} imagens extraídas\n')
        
    except Exception as e:
        arquivos_com_erro += 1
        print(f'  ❌ Erro ao processar arquivo: {e}\n')

print(f'\n{"="*60}')
print(f'✅ Arquivos processados: {arquivos_processados}/{len(arquivos_xlsx)}')
print(f'❌ Arquivos com erro: {arquivos_com_erro}')
print(f'🖼️  Total de imagens extraídas: {total_imagens}')
print(f'📁 Salvas em: {output_dir}')

## Extrair imagens E textos das legendas

Extrai imagens e os textos que estão logo abaixo de cada foto.

## Extrair imagens de TODOS os arquivos XLSX com '7127'

Processa todos os arquivos Excel encontrados no diretório recursivamente.

In [24]:
from openpyxl import load_workbook
from pathlib import Path
from PIL import Image
import io
import json

# Configurações
xlsx_path = Path('/Volumes/ssd_externo/CEF Edital 2011/Laudos de avaliação simplificados/AGE 0394/7127.0394.017404_2012.01.01.01/7127.0394.017404_2012.01.01.01.xlsx')
output_dir = Path('/Users/fjcosta/Downloads/imagens_com_legendas')

# Número máximo de linhas consecutivas a verificar
MAX_LINHAS = 5

# Criar diretório de saída
output_dir.mkdir(parents=True, exist_ok=True)

# Carregar o arquivo Excel
print(f'Carregando: {xlsx_path.name}\n')
workbook = load_workbook(xlsx_path)

imagens_extraidas = 0
metadados = {}

for sheet_name in workbook.sheetnames:
    sheet = workbook[sheet_name]
    
    if hasattr(sheet, '_images') and sheet._images:
        print(f'📊 Planilha "{sheet_name}": {len(sheet._images)} imagens')
        
        for img_index, image in enumerate(sheet._images):
            try:
                # Extrair dados da imagem
                image_data = image._data()
                pil_image = Image.open(io.BytesIO(image_data))
                img_format = pil_image.format.lower() if pil_image.format else 'png'
                
                # Nome do arquivo
                image_name = f"{xlsx_path.stem}_{sheet_name}_img{img_index+1}.{img_format}"
                image_name = image_name.replace('/', '_').replace('\\', '_')
                image_path = output_dir / image_name
                
                # Salvar imagem
                pil_image.save(image_path)
                
                # Extrair legenda
                legenda = ""
                if hasattr(image, 'anchor') and hasattr(image.anchor, '_from'):
                    col = image.anchor._from.col
                    row = image.anchor._from.row
                    
                    print(f'  📍 Imagem {img_index+1} ancorada em: Coluna {col}, Linha {row}')
                    
                    # Ler linhas consecutivas com texto
                    textos = []
                    for offset in range(1, MAX_LINHAS + 1):
                        linha_atual = []
                        # Verificar colunas adjacentes
                        for col_offset in range(-1, 3):
                            try:
                                cell = sheet.cell(row=row + offset, column=col + col_offset)
                                if cell.value and str(cell.value).strip():
                                    texto = str(cell.value).strip()
                                    if texto not in linha_atual:
                                        linha_atual.append(texto)
                            except:
                                pass
                        
                        if linha_atual:
                            textos.extend(linha_atual)
                        else:
                            # Para de buscar ao encontrar linha vazia
                            break
                    
                    if textos:
                        legenda = " ".join(textos)
                        print(f'  💬 Legenda: {legenda}')
                    else:
                        print(f'  ⚠️ Nenhum texto encontrado abaixo da imagem')
                
                # Salvar metadados
                metadados[image_name] = {
                    'planilha': sheet_name,
                    'legenda': legenda,
                    'dimensoes': f'{pil_image.width}x{pil_image.height}px'
                }
                
                imagens_extraidas += 1
                print()
                
            except Exception as e:
                print(f'  ❌ Erro ao processar imagem: {e}\n')

workbook.close()

# Salvar metadados em arquivo JSON
metadata_file = output_dir / f'{xlsx_path.stem}_legendas.json'
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

# Salvar também em arquivo TXT legível
txt_file = output_dir / f'{xlsx_path.stem}_legendas.txt'
with open(txt_file, 'w', encoding='utf-8') as f:
    for img_name, info in metadados.items():
        f.write(f'{img_name}\n')
        f.write(f'  Planilha: {info["planilha"]}\n')
        f.write(f'  Legenda: {info["legenda"]}\n')
        f.write(f'  Dimensões: {info["dimensoes"]}\n')
        f.write('\n')

print(f'{"="*60}')
print(f'✅ Total de imagens extraídas: {imagens_extraidas}')
print(f'📁 Imagens salvas em: {output_dir}')
print(f'📄 Legendas salvas em: {metadata_file.name} e {txt_file.name}')

Carregando: 7127.0394.017404_2012.01.01.01.xlsx

📊 Planilha "Levantamento  Fotográfico (1)": 4 imagens
  📍 Imagem 1 ancorada em: Coluna 6, Linha 5
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 2 ancorada em: Coluna 1, Linha 5
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 3 ancorada em: Coluna 6, Linha 18
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 4 ancorada em: Coluna 1, Linha 20
  ⚠️ Nenhum texto encontrado abaixo da imagem

📊 Planilha "Levantamento  Fotográfico (2)": 5 imagens
  📍 Imagem 1 ancorada em: Coluna 6, Linha 4
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 2 ancorada em: Coluna 1, Linha 4
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 3 ancorada em: Coluna 3, Linha 17
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 4 ancorada em: Coluna 1, Linha 17
  ⚠️ Nenhum texto encontrado abaixo da imagem

  📍 Imagem 5 ancorada em: Coluna 7, Linha 17
  ⚠️ Nenhum texto encontrado abaixo da imagem

📊 Planilha "Levanta